In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
## loading the trained ANN model, scaler pickle and OHE pickle
model = load_model('model.h5')

##loading the encoder and the scaler
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

# loading geography encder
with open('ohe_geo.pkl', 'rb') as file:
    ohe_geo = pickle.load(file)

## loading the scaler
with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [21]:
## Example input data
input_data = {
    'CreditScore' : 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalary' : 500000
}

## then convert this input to dataframe
input_df = pd.DataFrame([input_data])
input_df



,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,500000


### Now to this input data we also need to perform categorical encoding to convert the categorical features into numerical features
### Then we also need to apply standard scaling to do prediction

In [22]:
## apply OHE to geography
geo_encoded = ohe_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = ohe_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [23]:
## now concatinate these columns witht the original data
input_df = pd.concat([input_df.drop(['Geography'], axis = 1), geo_encoded_df], axis = 1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,500000,1.0,0.0,0.0


In [24]:
## applying label encoding to gender input
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,500000,1.0,0.0,0.0


In [26]:
### scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699,  6.94716943,  1.00150113,
        -0.57946723, -0.57638802]])

In [28]:
### predict churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step


array([[0.04046131]], dtype=float32)

In [30]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.040461313)

In [31]:
if prediction_proba > 0.5:
    print("The customer is likely to churn")
else:
    print("The customer will not churn")

The customer will not churn
